# MI Anatomical Localization Label Preparation

In [1]:
from __future__ import annotations

import ast
import json
import logging
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)
logger = logging.getLogger("ptbxl_mi_localization")


## Step 1: Configure paths

In [7]:
CURRENT_DIR = Path.cwd()

PROJECT_ROOT = next(
    (
        parent
        for parent in [CURRENT_DIR, *CURRENT_DIR.parents]
        if (parent / "data" / "raw" / "ptb-xl").is_dir()
    ),
    None
)

def find_existing_path(candidates: list[Path], description: str) -> Path:
    """Return the first existing candidate path or raise a helpful error."""
    for candidate in candidates:
        if candidate.exists():
            logger.info("Found %s: %s", description, candidate.resolve())
            return candidate
    candidate_text = "\\n".join(f"  - {path}" for path in candidates)
    raise FileNotFoundError(
        f"Could not find {description}. Checked:\\n{candidate_text}\\n"
        "Edit the path candidates in this cell to match your project."
    )

METADATA_PATH = find_existing_path(
    [
        PROJECT_ROOT / "data" / "ptbxl_database.csv",
        PROJECT_ROOT / "ptbxl_database.csv",
        PROJECT_ROOT / "data" / "raw" / "ptbxl_database.csv",
        PROJECT_ROOT / "data" / "raw" / "ptb-xl" / "ptbxl_database.csv",
    ],
    "ptbxl_database.csv",
)

SCP_PATH = find_existing_path(
    [
        PROJECT_ROOT / "data" / "scp_statements.csv",
        PROJECT_ROOT / "scp_statements.csv",
        PROJECT_ROOT / "data" / "raw" / "scp_statements.csv",
        PROJECT_ROOT / "data" / "raw" / "ptb-xl" / "scp_statements.csv",
    ],
    "scp_statements.csv",
)

OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "localization"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

logger.info("Outputs will be saved to: %s", OUTPUT_DIR.resolve())

2026-11-19 11:26:39,348 | INFO | Found ptbxl_database.csv: C:\Users\sintu\Desktop\Myocardial Infarction\data\raw\ptb-xl\ptbxl_database.csv
2026-11-19 11:26:39,350 | INFO | Found scp_statements.csv: C:\Users\sintu\Desktop\Myocardial Infarction\data\raw\ptb-xl\scp_statements.csv
2026-11-19 11:26:39,352 | INFO | Outputs will be saved to: C:\Users\sintu\Desktop\Myocardial Infarction\data\processed\localization


## Step 2: Load and validate PTB-XL metadata and SCP statements

In [8]:
metadata = pd.read_csv(METADATA_PATH)
scp_statements = pd.read_csv(SCP_PATH, index_col=0)

required_metadata_columns = {"ecg_id", "patient_id", "scp_codes"}
missing_metadata = required_metadata_columns.difference(metadata.columns)
if missing_metadata:
    raise ValueError(f"Metadata is missing required columns: {sorted(missing_metadata)}")

if scp_statements.index.name is None:
    scp_statements.index.name = "scp_code"

logger.info("Metadata shape: %s", metadata.shape)
logger.info("SCP statements shape: %s", scp_statements.shape)
display(metadata.head())
display(scp_statements.head())


2026-11-19 11:26:45,676 | INFO | Metadata shape: (21799, 28)
2026-11-19 11:26:45,677 | INFO | SCP statements shape: (71, 12)


,ecg_id,patient_id,age,sex,height,weight,nurse,site,device,recording_date,...,validated_by_human,baseline_drift,static_noise,burst_noise,electrodes_problems,extra_beats,pacemaker,strat_fold,filename_lr,filename_hr
0,1,15709.0,56.0,1,NaN,63.0,2.0,0.0,CS-12 E,1984-11-09 09:17:34,...,True,NaN,", I-V1,",NaN,NaN,NaN,NaN,3,records100/00000/00001_lr,records500/00000/00001_hr
1,2,13243.0,19.0,0,NaN,70.0,2.0,0.0,CS-12 E,1984-11-14 12:55:37,...,True,NaN,NaN,NaN,NaN,NaN,NaN,2,records100/00000/00002_lr,records500/00000/00002_hr
2,3,20372.0,37.0,1,NaN,69.0,2.0,0.0,CS-12 E,1984-11-15 12:49:10,...,True,NaN,NaN,NaN,NaN,NaN,NaN,5,records100/00000/00003_lr,records500/00000/00003_hr
3,4,17014.0,24.0,0,NaN,82.0,2.0,0.0,CS-12 E,1984-11-15 13:44:57,...,True,", II,III,AVF",NaN,NaN,NaN,NaN,NaN,3,records100/00000/00004_lr,records500/00000/00004_hr
4,5,17448.0,19.0,1,NaN,70.0,2.0,0.0,CS-12 E,1984-11-17 10:43:15,...,True,", III,AVR,AVF",NaN,NaN,NaN,NaN,NaN,4,records100/00000/00005_lr,records500/00000/00005_hr


,description,diagnostic,form,rhythm,diagnostic_class,diagnostic_subclass,Statement Category,SCP-ECG Statement Description,AHA code,aECG REFID,CDISC Code,DICOM Code
scp_code,,,,,,,,,,,,
NDT,non-diagnostic T abnormalities,1.0,1.0,NaN,STTC,STTC,other ST-T descriptive statements,non-diagnostic T abnormalities,NaN,NaN,NaN,NaN
NST_,non-specific ST changes,1.0,1.0,NaN,STTC,NST_,Basic roots for coding ST-T changes and abnorm...,non-specific ST changes,145.0,MDC_ECG_RHY_STHILOST,NaN,NaN
DIG,digitalis-effect,1.0,1.0,NaN,STTC,STTC,other ST-T descriptive statements,suggests digitalis-effect,205.0,NaN,NaN,NaN
LNGQT,long QT-interval,1.0,1.0,NaN,STTC,STTC,other ST-T descriptive statements,long QT-interval,148.0,NaN,NaN,NaN
NORM,normal ECG,1.0,NaN,NaN,NORM,NORM,Normal/abnormal,normal ECG,1.0,NaN,NaN,F-000B7


## Step 3: Parse SCP code dictionaries safely

In [9]:
def parse_scp_codes(value: Any) -> dict[str, float]:
    """Parse an SCP-code cell into {code: likelihood} safely."""
    if isinstance(value, dict):
        parsed = value
    elif pd.isna(value):
        return {}
    elif isinstance(value, str):
        try:
            parsed = ast.literal_eval(value)
        except (ValueError, SyntaxError) as exc:
            raise ValueError(f"Could not parse scp_codes value: {value!r}") from exc
    else:
        raise TypeError(f"Unsupported scp_codes value type: {type(value).__name__}")

    if not isinstance(parsed, dict):
        raise ValueError(f"Expected an SCP-code dictionary, got {type(parsed).__name__}")

    output: dict[str, float] = {}
    for code, likelihood in parsed.items():
        try:
            output[str(code)] = float(likelihood)
        except (TypeError, ValueError):
            logger.warning("Skipping non-numeric likelihood for SCP code %r", code)
    return output

metadata = metadata.copy()
metadata["scp_codes_dict"] = metadata["scp_codes"].apply(parse_scp_codes)
metadata["scp_code_set"] = metadata["scp_codes_dict"].apply(lambda item: set(item))
logger.info("Parsed SCP dictionaries for %d ECGs.", len(metadata))


2026-11-19 11:27:18,377 | INFO | Parsed SCP dictionaries for 21799 ECGs.


## Step 4: Identify MI-related diagnostic SCP statements

In [10]:
diagnostic_col = "diagnostic"
if diagnostic_col not in scp_statements.columns:
    raise ValueError(
        "scp_statements.csv does not contain a 'diagnostic' column. "
        f"Available columns: {list(scp_statements.columns)}"
    )

diagnostic_mask = pd.to_numeric(
    scp_statements[diagnostic_col], errors="coerce"
).fillna(0).astype(bool)

diagnostic_statements = scp_statements.loc[diagnostic_mask].copy()

if "diagnostic_class" in diagnostic_statements.columns:
    mi_mask = diagnostic_statements["diagnostic_class"].astype(str).str.upper().eq("MI")
elif "diagnostic_subclass" in diagnostic_statements.columns:
    mi_mask = diagnostic_statements["diagnostic_subclass"].astype(str).str.upper().str.contains(
        r"\\bMI\\b|INFARCT", regex=True, na=False
    )
else:
    raise ValueError(
        "Neither diagnostic_class nor diagnostic_subclass is available. "
        "Inspect the SCP table and define the MI statement filter explicitly."
    )

mi_statement_table = diagnostic_statements.loc[mi_mask].copy()
logger.info("MI diagnostic statements identified: %d", len(mi_statement_table))
display(mi_statement_table)


2026-11-19 11:27:47,162 | INFO | MI diagnostic statements identified: 14


,description,diagnostic,form,rhythm,diagnostic_class,diagnostic_subclass,Statement Category,SCP-ECG Statement Description,AHA code,aECG REFID,CDISC Code,DICOM Code
scp_code,,,,,,,,,,,,
IMI,inferior myocardial infarction,1.0,NaN,NaN,MI,IMI,Myocardial Infarction,inferior myocardial infarction,161.0,NaN,NaN,NaN
ASMI,anteroseptal myocardial infarction,1.0,NaN,NaN,MI,AMI,Myocardial Infarction,anteroseptal myocardial infarction,165.0,NaN,NaN,NaN
ILMI,inferolateral myocardial infarction,1.0,NaN,NaN,MI,IMI,Myocardial Infarction,inferolateral myocardial infarction,NaN,NaN,NaN,NaN
AMI,anterior myocardial infarction,1.0,NaN,NaN,MI,AMI,Myocardial Infarction,anterior myocardial infarction,160.0,NaN,NaN,NaN
ALMI,anterolateral myocardial infarction,1.0,NaN,NaN,MI,AMI,Myocardial Infarction,anterolateral myocardial infarction,NaN,NaN,NaN,NaN
INJAS,subendocardial injury in anteroseptal leads,1.0,NaN,NaN,MI,AMI,ischemic ST-T changes compatible with subendoc...,in anteroseptal leads,NaN,NaN,NaN,NaN
LMI,lateral myocardial infarction,1.0,NaN,NaN,MI,LMI,Myocardial Infarction,lateral myocardial infarction,163.0,NaN,NaN,NaN
INJAL,subendocardial injury in anterolateral leads,1.0,NaN,NaN,MI,AMI,ischemic ST-T changes compatible with subendoc...,in anterolateral leads,NaN,NaN,NaN,NaN
IPLMI,inferoposterolateral myocardial infarction,1.0,NaN,NaN,MI,IMI,Myocardial Infarction,inferoposterolateral myocardial infarction,NaN,NaN,NaN,NaN


## Step 5: Review MI SCP-code frequencies in this dataset

In [11]:
from collections import Counter

mi_codes = set(mi_statement_table.index.astype(str))
mi_code_counts: Counter[str] = Counter()

for code_set in metadata["scp_code_set"]:
    mi_code_counts.update(code for code in code_set if code in mi_codes)

mi_frequency = pd.DataFrame(
    [{"scp_code": code, "record_count": count} for code, count in mi_code_counts.most_common()]
)

if not mi_frequency.empty:
    mi_frequency = mi_frequency.merge(
        mi_statement_table.reset_index(),
        left_on="scp_code",
        right_on=mi_statement_table.index.name or "index",
        how="left",
    )

display(mi_frequency)


,scp_code,record_count,description,diagnostic,form,rhythm,diagnostic_class,diagnostic_subclass,Statement Category,SCP-ECG Statement Description,AHA code,aECG REFID,CDISC Code,DICOM Code
0,IMI,2676,inferior myocardial infarction,1.0,NaN,NaN,MI,IMI,Myocardial Infarction,inferior myocardial infarction,161.0,NaN,NaN,NaN
1,ASMI,2357,anteroseptal myocardial infarction,1.0,NaN,NaN,MI,AMI,Myocardial Infarction,anteroseptal myocardial infarction,165.0,NaN,NaN,NaN
2,ILMI,478,inferolateral myocardial infarction,1.0,NaN,NaN,MI,IMI,Myocardial Infarction,inferolateral myocardial infarction,NaN,NaN,NaN,NaN
3,AMI,353,anterior myocardial infarction,1.0,NaN,NaN,MI,AMI,Myocardial Infarction,anterior myocardial infarction,160.0,NaN,NaN,NaN
4,ALMI,288,anterolateral myocardial infarction,1.0,NaN,NaN,MI,AMI,Myocardial Infarction,anterolateral myocardial infarction,NaN,NaN,NaN,NaN
5,INJAS,214,subendocardial injury in anteroseptal leads,1.0,NaN,NaN,MI,AMI,ischemic ST-T changes compatible with subendoc...,in anteroseptal leads,NaN,NaN,NaN,NaN
6,LMI,201,lateral myocardial infarction,1.0,NaN,NaN,MI,LMI,Myocardial Infarction,lateral myocardial infarction,163.0,NaN,NaN,NaN
7,INJAL,145,subendocardial injury in anterolateral leads,1.0,NaN,NaN,MI,AMI,ischemic ST-T changes compatible with subendoc...,in anterolateral leads,NaN,NaN,NaN,NaN
8,IPLMI,51,inferoposterolateral myocardial infarction,1.0,NaN,NaN,MI,IMI,Myocardial Infarction,inferoposterolateral myocardial infarction,NaN,NaN,NaN,NaN
9,IPMI,33,inferoposterior myocardial infarction,1.0,NaN,NaN,MI,IMI,Myocardial Infarction,inferoposterior myocardial infarction,NaN,NaN,NaN,NaN


## Step 6: Define an explicit SCP-code-to-territory mapping

In [12]:
TERRITORY_NAMES = [
    "inferior",
    "anterior",
    "anteroseptal",
    "anterolateral",
    "lateral",
    "posterior",
    "inferoposterior",
]

SCP_TO_TERRITORIES: dict[str, list[str]] = {
    "IMI": ["inferior"],
    "AMI": ["anterior"],
    "ASMI": ["anteroseptal"],
    "ALMI": ["anterolateral"],
    "ILMI": ["inferior", "lateral"],
    "IPLMI": ["inferior", "posterior", "lateral"],
    "IPMI": ["inferior", "posterior"],
    "PMI": ["posterior"],
}

mapped_codes = set(SCP_TO_TERRITORIES)
observed_mi_codes = set(mi_frequency["scp_code"]) if not mi_frequency.empty else set()

unmapped_observed_codes = sorted(observed_mi_codes - mapped_codes)
mapping_codes_not_in_reference = sorted(mapped_codes - mi_codes)

print("Observed MI codes without a mapping:", unmapped_observed_codes)
print("Mapped codes absent from the MI reference table:", mapping_codes_not_in_reference)

for code, territories in SCP_TO_TERRITORIES.items():
    invalid = set(territories) - set(TERRITORY_NAMES)
    if invalid:
        raise ValueError(f"Invalid territories for {code}: {sorted(invalid)}")


Observed MI codes without a mapping: ['INJAL', 'INJAS', 'INJIL', 'INJIN', 'INJLA', 'LMI']
Mapped codes absent from the MI reference table: []


## Step 7: Build multi-hot localization labels

In [13]:
def map_record_to_territories(code_dict: dict[str, float]) -> tuple[list[str], list[str]]:
    """Return mapped territories and observed but unmapped MI codes."""
    observed_mi = sorted(set(code_dict).intersection(mi_codes))
    territories: set[str] = set()
    unmapped: list[str] = []

    for code in observed_mi:
        if code in SCP_TO_TERRITORIES:
            territories.update(SCP_TO_TERRITORIES[code])
        else:
            unmapped.append(code)

    return sorted(territories), unmapped

mapped = metadata["scp_codes_dict"].apply(map_record_to_territories)
metadata["mi_territories"] = mapped.apply(lambda item: item[0])
metadata["unmapped_mi_codes"] = mapped.apply(lambda item: item[1])
metadata["has_mi_scp"] = metadata["scp_code_set"].apply(lambda codes: bool(codes.intersection(mi_codes)))
metadata["localization_mapping_complete"] = metadata.apply(
    lambda row: (not row["has_mi_scp"]) or len(row["unmapped_mi_codes"]) == 0,
    axis=1,
)

for territory in TERRITORY_NAMES:
    metadata[f"loc_{territory}"] = metadata["mi_territories"].apply(
        lambda labels: int(territory in labels)
    )

target_columns = [f"loc_{territory}" for territory in TERRITORY_NAMES]
logger.info("Records with any MI SCP code: %d", int(metadata["has_mi_scp"].sum()))
logger.info(
    "MI records with complete mapping: %d",
    int((metadata["has_mi_scp"] & metadata["localization_mapping_complete"]).sum()),
)
display(metadata[["ecg_id", "patient_id", "has_mi_scp", "mi_territories",
                  "unmapped_mi_codes", *target_columns]].head(10))


2026-11-19 11:29:08,138 | INFO | Records with any MI SCP code: 5469
2026-11-19 11:29:08,139 | INFO | MI records with complete mapping: 4953


,ecg_id,patient_id,has_mi_scp,mi_territories,unmapped_mi_codes,loc_inferior,loc_anterior,loc_anteroseptal,loc_anterolateral,loc_lateral,loc_posterior,loc_inferoposterior
0,1,15709.0,False,[],[],0,0,0,0,0,0,0
1,2,13243.0,False,[],[],0,0,0,0,0,0,0
2,3,20372.0,False,[],[],0,0,0,0,0,0,0
3,4,17014.0,False,[],[],0,0,0,0,0,0,0
4,5,17448.0,False,[],[],0,0,0,0,0,0,0
5,6,19005.0,False,[],[],0,0,0,0,0,0,0
6,7,16193.0,False,[],[],0,0,0,0,0,0,0
7,8,11275.0,True,[inferior],[],1,0,0,0,0,0,0
8,9,18792.0,False,[],[],0,0,0,0,0,0,0
9,10,9456.0,False,[],[],0,0,0,0,0,0,0


## Step 8: Audit class frequencies and overlapping labels

In [14]:
mi_rows = metadata.loc[metadata["has_mi_scp"]].copy()
complete_mi_rows = mi_rows.loc[mi_rows["localization_mapping_complete"]].copy()

territory_counts = complete_mi_rows[target_columns].sum().sort_values(ascending=False)
territory_summary = pd.DataFrame({
    "target": territory_counts.index,
    "positive_records": territory_counts.values,
})
territory_summary["positive_fraction_among_complete_mi"] = (
    territory_summary["positive_records"] / max(len(complete_mi_rows), 1)
)

print(f"All records: {len(metadata):,}")
print(f"MI-coded records: {len(mi_rows):,}")
print(f"MI-coded records with complete mapping: {len(complete_mi_rows):,}")
print(f"MI-coded records with unmapped MI codes: {(~mi_rows['localization_mapping_complete']).sum():,}")
print(f"Records with more than one territory: {(complete_mi_rows[target_columns].sum(axis=1) > 1).sum():,}")

display(territory_summary)


All records: 21,799
MI-coded records: 5,469
MI-coded records with complete mapping: 4,953
MI-coded records with unmapped MI codes: 516
Records with more than one territory: 1,408


,target,positive_records,positive_fraction_among_complete_mi
0,loc_inferior,3157,0.637391
1,loc_anteroseptal,2226,0.449425
2,loc_lateral,521,0.105189
3,loc_anterior,330,0.066626
4,loc_anterolateral,287,0.057945
5,loc_posterior,97,0.019584
6,loc_inferoposterior,0,0.000000


## Step 9: Attach the existing official split assignment

In [15]:
existing_labels_candidates = [
    PROJECT_ROOT / "data" / "processed" / "ptbxl_labels.csv",
    PROJECT_ROOT / "data" / "processed" / "labels" / "ptbxl_labels.csv",
    PROJECT_ROOT / "ptbxl_labels.csv",
]
existing_labels_path = next((p for p in existing_labels_candidates if p.exists()), None)

if existing_labels_path is not None:
    existing_labels = pd.read_csv(existing_labels_path)
    if "ecg_id" not in existing_labels.columns:
        if existing_labels.index.name == "ecg_id":
            existing_labels = existing_labels.reset_index()
        else:
            raise ValueError(
                f"{existing_labels_path} must contain ecg_id as a column or named index."
            )

    split_candidates = ["dataset_split", "split", "set"]
    split_col = next((col for col in split_candidates if col in existing_labels.columns), None)

    if split_col is None:
        logger.warning(
            "Existing labels file found but no split column was found. "
            "Will use PTB-XL strat_fold fallback."
        )
        split_map = None
    else:
        split_map = existing_labels[["ecg_id", split_col]].drop_duplicates("ecg_id")
        split_map = split_map.rename(columns={split_col: "dataset_split"})
        logger.info("Using split assignments from %s", existing_labels_path)
else:
    split_map = None

if split_map is not None:
    metadata = metadata.merge(split_map, on="ecg_id", how="left", validate="one_to_one")
    if metadata["dataset_split"].isna().any():
        missing_ids = metadata.loc[metadata["dataset_split"].isna(), "ecg_id"].head().tolist()
        raise ValueError(f"Missing split assignments for ECG IDs, examples: {missing_ids}")
else:
    if "strat_fold" not in metadata.columns:
        raise ValueError(
            "No existing split column was found and ptbxl_database.csv has no strat_fold column."
        )
    metadata["dataset_split"] = np.select(
        [
            metadata["strat_fold"].between(1, 8),
            metadata["strat_fold"].eq(9),
            metadata["strat_fold"].eq(10),
        ],
        ["train", "validation", "test"],
        default="excluded",
    )

print(metadata["dataset_split"].value_counts(dropna=False))


2026-11-19 11:30:01,112 | INFO | Using split assignments from c:\Users\sintu\Desktop\Myocardial Infarction\data\processed\ptbxl_labels.csv


dataset_split
train         17418
test           2198
validation     2183
Name: count, dtype: int64


## Step 10: Patient leakage and split integrity checks

In [16]:
def validate_patient_splits(frame: pd.DataFrame) -> None:
    """Raise if a patient appears in more than one active split."""
    active = frame.loc[frame["dataset_split"].isin(["train", "validation", "val", "test"])]
    split_counts = active.groupby("patient_id")["dataset_split"].nunique()
    leaking_patients = split_counts[split_counts > 1]
    if not leaking_patients.empty:
        raise ValueError(
            f"Patient leakage detected for {len(leaking_patients)} patients. "
            f"Examples: {leaking_patients.index[:10].tolist()}"
        )
    logger.info("Patient-level split check passed.")

validate_patient_splits(metadata)

split_summary = (
    metadata.groupby("dataset_split")
    .agg(
        recordings=("ecg_id", "count"),
        patients=("patient_id", "nunique"),
        mi_recordings=("has_mi_scp", "sum"),
    )
    .sort_index()
)
display(split_summary)


2026-11-19 11:30:36,524 | INFO | Patient-level split check passed.


,recordings,patients,mi_recordings
dataset_split,,,
test,2198,1904,550
train,17418,15023,4379
validation,2183,1942,540


## Step 11: Save localization targets and mapping audit artifacts

In [17]:
output_columns = [
    "ecg_id",
    "patient_id",
    "dataset_split",
    "has_mi_scp",
    "localization_mapping_complete",
    "mi_territories",
    "unmapped_mi_codes",
    *target_columns,
]
if "filename_hr" in metadata.columns:
    output_columns.insert(3, "filename_hr")

localization_targets = metadata[output_columns].copy()
localization_targets["mi_territories"] = localization_targets["mi_territories"].apply(
    lambda values: json.dumps(values)
)
localization_targets["unmapped_mi_codes"] = localization_targets["unmapped_mi_codes"].apply(
    lambda values: json.dumps(values)
)

targets_path = OUTPUT_DIR / "ptbxl_mi_localization_labels.csv"
mapping_path = OUTPUT_DIR / "scp_to_territory_mapping.json"
frequency_path = OUTPUT_DIR / "mi_scp_frequency_audit.csv"
summary_path = OUTPUT_DIR / "localization_label_summary.json"

localization_targets.to_csv(targets_path, index=False)
mi_frequency.to_csv(frequency_path, index=False)

mapping_payload = {
    "territories": TERRITORY_NAMES,
    "scp_to_territories": SCP_TO_TERRITORIES,
    "note": (
        "Research mapping requiring clinical review. PTB-XL SCP labels are ECG "
        "diagnostic statements and are not imaging-confirmed anatomical ground truth."
    ),
}
mapping_path.write_text(json.dumps(mapping_payload, indent=2), encoding="utf-8")

summary_payload = {
    "n_recordings": int(len(metadata)),
    "n_patients": int(metadata["patient_id"].nunique()),
    "n_mi_scp_recordings": int(metadata["has_mi_scp"].sum()),
    "n_complete_mi_mappings": int(
        (metadata["has_mi_scp"] & metadata["localization_mapping_complete"]).sum()
    ),
    "n_mi_records_with_unmapped_codes": int(
        (metadata["has_mi_scp"] & ~metadata["localization_mapping_complete"]).sum()
    ),
    "target_columns": target_columns,
    "split_counts": metadata["dataset_split"].value_counts().to_dict(),
}
summary_path.write_text(json.dumps(summary_payload, indent=2), encoding="utf-8")

print(f"Saved: {targets_path}")
print(f"Saved: {mapping_path}")
print(f"Saved: {frequency_path}")
print(f"Saved: {summary_path}")


Saved: c:\Users\sintu\Desktop\Myocardial Infarction\data\processed\localization\ptbxl_mi_localization_labels.csv
Saved: c:\Users\sintu\Desktop\Myocardial Infarction\data\processed\localization\scp_to_territory_mapping.json
Saved: c:\Users\sintu\Desktop\Myocardial Infarction\data\processed\localization\mi_scp_frequency_audit.csv
Saved: c:\Users\sintu\Desktop\Myocardial Infarction\data\processed\localization\localization_label_summary.json
